# Modelos avançados no Kaggle

Este notebook **não roda no ambiente local** — foi gerado aqui, mas é
feito para ser subido e executado no Kaggle, onde PyTorch + GPU já vêm
prontos (ver `reports/tables/modelos_avancados_smoke_test.md` para o
porquê de não dar para rodar isso localmente).

Segue a mesma organização dos outros notebooks deste repositório: o código
fica no pacote `src/churn_telecom/`, e este notebook só clona o repositório,
instala o pacote e orquestra os experimentos.

**No Kaggle, antes de rodar:** ative *Settings → Internet → On* (o
notebook clona o repositório, baixa o CSV e instala os pacotes) e
*Accelerator → GPU*. Depois use **"Save Version" → "Save & Run All
(Commit)"**, que executa num kernel gerenciado em segundo plano.


In [ ]:
#@title Setup (clona o repo e instala o pacote)
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/jpbezerra/redes-neurais.git"   # ajuste se necessario
BRANCH   = "feat/churn"   # ajuste aqui se a branch de trabalho mudar
REPO_DIR = "redes-neurais"
SUBPATH  = "projeto/churn-telecom"

def _sh(cmd):
    print("$", cmd); subprocess.run(cmd, shell=True, check=False)

IN_COLAB  = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle")

if IN_COLAB or IN_KAGGLE:
    base = Path("/content") if IN_COLAB else Path("/kaggle/working")
    root = base / REPO_DIR
    if root.exists():
        _sh(f"cd {root} && git fetch origin {BRANCH} && git checkout {BRANCH} && git pull --ff-only origin {BRANCH}")
    else:
        _sh(f"git clone --branch {BRANCH} --single-branch {REPO_URL} {root}")
    PROJECT_ROOT = root / SUBPATH
    _sh(f"pip install -q -e {PROJECT_ROOT}")
else:
    PROJECT_ROOT = Path.cwd().parent

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

DATA_DIR    = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
TABLES_DIR  = PROJECT_ROOT / "reports" / "tables"
for d in (DATA_DIR, RESULTS_DIR, TABLES_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RESULTS_DIR :", RESULTS_DIR)


## Instalação dos modelos avançados

O Kaggle já vem com PyTorch e a maioria das libs de ML — só falta instalar
os 5 modelos em si.

In [ ]:
!pip install -q tabpfn pykan tabkan
!pip install -q "autogluon.tabular[mitra]"


## Preparação dos dados

`load_raw` baixa o CSV automaticamente se não existir em `DATA_DIR`
(mesma conveniência do mini-projeto 2) — não é preciso subir nada como
Kaggle Dataset separado, o clone do repositório já basta.

In [ ]:
import torch
import joblib
import numpy as np
import pandas as pd

from churn_telecom.data import load_raw, split_three_stage, assert_sem_vazamento
from churn_telecom.features import Preprocessor
from churn_telecom.metrics import full_report
from churn_telecom.checkpointing import save_run, load_all_metadata

SEED = 0
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

df = load_raw(DATA_DIR)
split = split_three_stage(df, seed=SEED)
assert_sem_vazamento(split)
df_train, df_val, df_test = df.iloc[split.train_idx], df.iloc[split.val_idx], df.iloc[split.test_idx]

prep = Preprocessor().fit(df_train)
X_train, cols = prep.transform(df_train)
X_val, _ = prep.transform(df_val)
X_test, _ = prep.transform(df_test)
y_train = prep.transform_target(df_train)
y_val = prep.transform_target(df_val)
y_test = prep.transform_target(df_test)

print(f"X_train: {X_train.shape} | X_val: {X_val.shape} | X_test: {X_test.shape}")


## STab (implementação própria — ver `src/churn_telecom/models/stab.py`)

Não existe um pacote público legítimo chamado "STab" (colisão de nome no
PyPI — ver smoke test). Esta é uma implementação própria e enxuta de
Transformer para dados tabulares, cobrindo o espaço de hiperparâmetros
pedido no enunciado.

In [ ]:
from churn_telecom.models.stab import STabHyperparams, treinar_stab, prever_stab

hp_stab = STabHyperparams(dim=32, depth=2, heads=4, attn_dropout=0.1, ff_dropout=0.1,
                           lr=1e-3, weight_decay=1e-5, batch_size=64, max_epochs=200, patience=10, seed=SEED)
modelo_stab, historico_stab = treinar_stab(X_train, y_train, X_val, y_val, hp_stab, device=device, verbose=True)
p_test_stab = prever_stab(modelo_stab, X_test, device=device)
metricas_stab = full_report(y_test, p_test_stab)
print(metricas_stab)

def salvar_stab(save_dir):
    torch.save(modelo_stab.state_dict(), save_dir / "model.pt")

save_run(RESULTS_DIR, "stab_churn", framework="PyTorch (STab — implementacao propria)",
         hyperparameters=hp_stab.__dict__, metrics=metricas_stab,
         architecture={"dim": hp_stab.dim, "depth": hp_stab.depth, "heads": hp_stab.heads},
         history=historico_stab, notes="STab (implementacao propria).", save_model_fn=salvar_stab)


## TabPFN v2

In [ ]:
from churn_telecom.models.tabpfn import TabPFNHyperparams, treinar_tabpfn, prever_tabpfn

hp_tabpfn = TabPFNHyperparams(device=device, n_estimators=8, seed=SEED)
modelo_tabpfn = treinar_tabpfn(X_train, y_train, hp_tabpfn)
p_test_tabpfn = prever_tabpfn(modelo_tabpfn, X_test)
metricas_tabpfn = full_report(y_test, p_test_tabpfn)
print(metricas_tabpfn)

def salvar_tabpfn(save_dir):
    joblib.dump(modelo_tabpfn, save_dir / "model.joblib")

save_run(RESULTS_DIR, "tabpfn_churn", framework="TabPFN v2", hyperparameters=hp_tabpfn.__dict__,
         metrics=metricas_tabpfn, notes="TabPFN v2.", save_model_fn=salvar_tabpfn)


## KAN

In [ ]:
from churn_telecom.models.kan import KANHyperparams, treinar_kan, prever_kan

hp_kan = KANHyperparams(hidden_width=10, grid=5, k=3, lr=1e-3, steps=50, seed=SEED)
modelo_kan, historico_kan = treinar_kan(X_train, y_train, X_val, y_val, hp_kan, device=device)
p_test_kan = prever_kan(modelo_kan, X_test, device=device)
metricas_kan = full_report(y_test, p_test_kan)
print(metricas_kan)

def salvar_kan(save_dir):
    torch.save(modelo_kan.state_dict(), save_dir / "model.pt")

save_run(RESULTS_DIR, "kan_churn", framework="pykan (KAN)", hyperparameters=hp_kan.__dict__,
         metrics=metricas_kan, architecture={"hidden_width": hp_kan.hidden_width, "grid": hp_kan.grid, "k": hp_kan.k},
         notes="KAN.", save_model_fn=salvar_kan)


## TabKAN

**Atenção:** a API exata do pacote `tabkan` não pôde ser inspecionada no
ambiente local (não instala lá). Se `TabKANClassifier` não existir ou
tiver assinatura diferente, ajuste `src/churn_telecom/models/tabkan.py`
conforme a documentação real do pacote instalado aqui.

In [ ]:
from churn_telecom.models.tabkan import TabKANHyperparams, treinar_tabkan, prever_tabkan

hp_tabkan = TabKANHyperparams(hidden_width=10, grid=5, k=3, lr=1e-3, epochs=100, seed=SEED)
modelo_tabkan = treinar_tabkan(X_train, y_train, hp_tabkan)
p_test_tabkan = prever_tabkan(modelo_tabkan, X_test)
metricas_tabkan = full_report(y_test, p_test_tabkan)
print(metricas_tabkan)

def salvar_tabkan(save_dir):
    joblib.dump(modelo_tabkan, save_dir / "model.joblib")

save_run(RESULTS_DIR, "tabkan_churn", framework="tabkan", hyperparameters=hp_tabkan.__dict__,
         metrics=metricas_tabkan, notes="TabKAN.", save_model_fn=salvar_tabkan)


## Mitra (via AutoGluon)

In [ ]:
from churn_telecom.models.mitra import MitraHyperparams, treinar_mitra, prever_mitra

hp_mitra = MitraHyperparams(time_limit_segundos=600, seed=SEED)
preditor_mitra = treinar_mitra(X_train, y_train, cols, hp_mitra)
p_test_mitra = prever_mitra(preditor_mitra, X_test, cols)
metricas_mitra = full_report(y_test, p_test_mitra)
print(metricas_mitra)

def salvar_mitra(save_dir):
    preditor_mitra.save(path=str(save_dir / "autogluon_predictor"))

save_run(RESULTS_DIR, "mitra_churn", framework="AutoGluon (extra [mitra])",
         hyperparameters=hp_mitra.__dict__, metrics=metricas_mitra, notes="Mitra.",
         save_model_fn=salvar_mitra)


## Comparação final — todos os modelos do projeto

Esta célula também funciona se algum dos modelos acima falhar (está
isolado célula a célula) — basta ter rodado os que funcionaram.

In [ ]:
resumo = load_all_metadata(RESULTS_DIR)
cols_mostrar = ["model_id", "metrics.ks", "metrics.auroc", "metrics.recall", "metrics.precision"]
cols_existentes = [c for c in cols_mostrar if c in resumo.columns]
tabela = resumo[cols_existentes].sort_values("metrics.ks", ascending=False).reset_index(drop=True)
tabela


## Trazendo os resultados de volta

O "Save & Run All (Commit)" do Kaggle versiona os arquivos gerados em
`/kaggle/working`. Depois, baixe a pasta `results/` do commit e cole em
`results/` no projeto local, junto com qualquer CSV novo em
`reports/tables/` — o notebook de consolidação final só lê `results/`,
não retreina nada.
